# Day 2 — Runtime Architecture

**What you will learn:**
- The three components of a Spark cluster
- How a query becomes Jobs → Stages → Tasks
- Narrow vs wide transformations — why it matters
- Deployment modes: local, client, cluster
- How Spark recovers from failures

**Exam relevance:** Architecture (20%) — execution hierarchy and deployment modes are tested directly.

> **To activate interactive elements** (quiz + walkthrough): **Run All Cells** (`Ctrl+Alt+R` on Windows/Linux, `Cmd+Option+R` on Mac). The SVG diagrams render automatically.

## 1. Cluster Components

```
┌─────────────────────────────────────────────────┐
│                  SPARK CLUSTER                  │
│                                                 │
│  ┌──────────────┐       ┌───────────────────┐  │
│  │    DRIVER    │◄─────►│  CLUSTER MANAGER  │  │
│  │              │       │  (YARN/K8s/DBR)   │  │
│  │ SparkContext │       └───────────────────┘  │
│  │ Scheduler    │               │               │
│  │ DAG Builder  │       ┌───────┴───────┐       │
│  └──────────────┘       ▼               ▼       │
│                   ┌──────────┐  ┌──────────┐   │
│                   │EXECUTOR 1│  │EXECUTOR 2│   │
│                   │ Task Task│  │ Task Task│   │
│                   │ [cache]  │  │ [cache]  │   │
│                   └──────────┘  └──────────┘   │
└─────────────────────────────────────────────────┘
```

| Component | Role |
|---|---|
| **Driver** | Your notebook / application. Builds the execution plan, schedules tasks, collects results. |
| **Cluster Manager** | Allocates resources (workers). Options: Databricks, YARN, Kubernetes, Mesos, Standalone. |
| **Executors** | JVM processes on worker nodes. Run tasks, store cached data. Created at app start, live for its duration. |

In [ ]:
displayHTML("""<div style="min-height:390px;"><svg width="100%" viewBox="0 0 711.25 360" role="img" style="" xmlns="http://www.w3.org/2000/svg">
  <title style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">Spark architecture: Driver, Cluster Manager, Executors</title>
  <desc style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">The Driver sends tasks to Executors via the Cluster Manager. Executors run on Worker nodes and send results back to the Driver.</desc>
  <defs>
    <marker id="arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  <mask id="imagine-text-gaps-dk49qp" maskUnits="userSpaceOnUse"><rect x="0" y="0" width="711.25" height="360" fill="white"/><rect x="315.09375" y="44.234375" width="49.8125" height="21.53125" fill="black" rx="2"/><rect x="255.671875" y="65.484375" width="168.65625" height="19.03125" fill="black" rx="2"/><rect x="280.421875" y="162.234375" width="119.15625" height="21.515625" fill="black" rx="2"/><rect x="265.625" y="183.484375" width="148.75" height="19.03125" fill="black" rx="2"/><rect x="80.03125" y="281.25" width="139.9375" height="21.515625" fill="black" rx="2"/><rect x="78.28125" y="302.484375" width="143.4375" height="19.015625" fill="black" rx="2"/><rect x="458.609375" y="281.25" width="142.78125" height="21.515625" fill="black" rx="2"/><rect x="458.28125" y="302.484375" width="143.4375" height="19.015625" fill="black" rx="2"/><rect x="238.28125" y="114.484375" width="105.4375" height="19.015625" fill="black" rx="2"/><rect x="162.171875" y="232.078125" width="45.65625" height="19.015625" fill="black" rx="2"/><rect x="472.171875" y="232.078125" width="45.65625" height="19.015625" fill="black" rx="2"/><rect x="42.640625" y="154.078125" width="38.71875" height="19.015625" fill="black" rx="2"/><rect x="-25.25" y="186.078125" width="47.25" height="19.015625" fill="black" rx="2"/><rect x="598.640625" y="154.078125" width="38.71875" height="19.015625" fill="black" rx="2"/><rect x="658" y="186.078125" width="47.25" height="19.015625" fill="black" rx="2"/></mask></defs>

  <!-- Driver -->
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="240" y="30" width="200" height="64" rx="8" stroke-width="0.5" style="fill:rgb(238, 237, 254);stroke:rgb(83, 74, 183);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="340" y="55" text-anchor="middle" dominant-baseline="central" style="fill:rgb(60, 52, 137);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Driver</text>
    <text x="340" y="75" text-anchor="middle" dominant-baseline="central" style="fill:rgb(83, 74, 183);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">Builds DAG, schedules tasks</text>
  </g>

  <!-- Cluster Manager -->
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="240" y="148" width="200" height="64" rx="8" stroke-width="0.5" style="fill:rgb(225, 245, 238);stroke:rgb(15, 110, 86);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="340" y="173" text-anchor="middle" dominant-baseline="central" style="fill:rgb(8, 80, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Cluster manager</text>
    <text x="340" y="193" text-anchor="middle" dominant-baseline="central" style="fill:rgb(15, 110, 86);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">Allocates CPU &amp; memory</text>
  </g>

  <!-- Worker 1 + Executor -->
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="60" y="270" width="180" height="64" rx="8" stroke-width="0.5" style="fill:rgb(250, 236, 231);stroke:rgb(153, 60, 29);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="150" y="292" text-anchor="middle" dominant-baseline="central" style="fill:rgb(113, 43, 19);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Executor (worker 1)</text>
    <text x="150" y="312" text-anchor="middle" dominant-baseline="central" style="fill:rgb(153, 60, 29);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">Runs tasks, caches data</text>
  </g>

  <!-- Worker 2 + Executor -->
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="440" y="270" width="180" height="64" rx="8" stroke-width="0.5" style="fill:rgb(250, 236, 231);stroke:rgb(153, 60, 29);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="530" y="292" text-anchor="middle" dominant-baseline="central" style="fill:rgb(113, 43, 19);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Executor (worker 2)</text>
    <text x="530" y="312" text-anchor="middle" dominant-baseline="central" style="fill:rgb(153, 60, 29);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">Runs tasks, caches data</text>
  </g>

  <!-- Driver → Cluster Manager: resource request -->
  <line x1="340" y1="94" x2="340" y2="146" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" mask="url(#imagine-text-gaps-dk49qp)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <rect x="244" y="112" width="94" height="18" rx="4" fill="#f5f4ed" style="fill:rgb(245, 244, 237);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="291" y="124" text-anchor="middle" dominant-baseline="central" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">resource request</text>

  <!-- Cluster Manager → Executor 1 -->
  <line x1="270" y1="212" x2="190" y2="268" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="185" y="246" text-anchor="middle" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:auto">launch</text>

  <!-- Cluster Manager → Executor 2 -->
  <line x1="410" y1="212" x2="490" y2="268" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="495" y="246" text-anchor="middle" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:auto">launch</text>

  <!-- Driver ↔ Executor 1: tasks / results -->
  <path d="M240 80 Q100 80 100 268" fill="none" stroke="#3d3d3a" stroke-width="1" stroke-dasharray="5 3" marker-end="url(#arrow)" style="fill:none;stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-dasharray:5px, 3px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="62" y="168" text-anchor="middle" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:auto">tasks</text>
  <path d="M60 302 Q20 302 20 80 Q20 62 60 62" fill="none" stroke="#3d3d3a" stroke-width="1" stroke-dasharray="5 3" marker-end="url(#arrow)" style="fill:none;stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-dasharray:5px, 3px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="18" y="200" text-anchor="end" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:end;dominant-baseline:auto">results</text>

  <!-- Driver ↔ Executor 2: tasks / results -->
  <path d="M440 80 Q580 80 580 268" fill="none" stroke="#3d3d3a" stroke-width="1" stroke-dasharray="5 3" marker-end="url(#arrow)" style="fill:none;stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-dasharray:5px, 3px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="618" y="168" text-anchor="middle" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:auto">tasks</text>
  <path d="M620 302 Q660 302 660 80 Q660 62 620 62" fill="none" stroke="#3d3d3a" stroke-width="1" stroke-dasharray="5 3" marker-end="url(#arrow)" style="fill:none;stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-dasharray:5px, 3px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="662" y="200" text-anchor="start" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:start;dominant-baseline:auto">results</text>
</svg></div>
<script>try{window.frameElement.style.height=(document.body.scrollHeight+20)+"px";}catch(e){}</script>""")

In [ ]:
# sparkContext JVM access is not available on Databricks serverless
try:
    sc = spark.sparkContext
    print(f"Master             : {sc.master}")
    print(f"Default parallelism: {sc.defaultParallelism}")  # total cores across all executors
    print(f"App name           : {sc.appName}")
except Exception:
    print("Note: sparkContext JVM attributes unavailable on serverless compute.")
    print("On a classic cluster these return the master URL, total cores, and app name.")
    print(f"Spark version: {spark.version}  — session is active and working.")

## 2. Jobs → Stages → Tasks

When you call an **action** (`.show()`, `.count()`, `.write()`), Spark creates a **Job**.  
Each job is broken into **Stages** at shuffle boundaries.  
Each stage is split into **Tasks** — one task per partition.

```
ACTION called
    └── JOB
         ├── STAGE 1  (narrow transformations — no shuffle)
         │     ├── Task (partition 0)
         │     ├── Task (partition 1)
         │     └── Task (partition N)
         └── STAGE 2  (after shuffle boundary)
               ├── Task (partition 0)
               └── Task (partition 1)
```

> **Rule:** Every shuffle creates a new stage boundary.

In [ ]:
displayHTML("""<div style="min-height:410px;"><svg width="100%" viewBox="0 0 680 380" role="img" style="" xmlns="http://www.w3.org/2000/svg">
  <title style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">Spark execution model: Action triggers Job, split into Stages, each Stage into Tasks</title>
  <desc style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">An action like count() triggers a Job. A shuffle boundary splits the Job into Stages. Each Stage is divided into Tasks, one per partition.</desc>
  <defs>
    <marker id="arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="context-stroke" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  <mask id="imagine-text-gaps-81iyo9" maskUnits="userSpaceOnUse"><rect x="0" y="0" width="680" height="380" fill="white"/><rect x="313.71875" y="31.234375" width="52.5625" height="21.53125" fill="black" rx="2"/><rect x="258.671875" y="50.484375" width="162.65625" height="19.03125" fill="black" rx="2"/><rect x="348" y="83.484375" width="52.515625" height="19.03125" fill="black" rx="2"/><rect x="323.65625" y="115.234375" width="32.6875" height="21.53125" fill="black" rx="2"/><rect x="254.5" y="134.484375" width="171" height="19.03125" fill="black" rx="2"/><rect x="348" y="167.484375" width="92.484375" height="19.03125" fill="black" rx="2"/><rect x="147.609375" y="199.234375" width="54.78125" height="21.515625" fill="black" rx="2"/><rect x="98.34375" y="218.484375" width="153.3125" height="19.03125" fill="black" rx="2"/><rect x="476.203125" y="199.234375" width="57.59375" height="21.515625" fill="black" rx="2"/><rect x="430.453125" y="218.484375" width="149.09375" height="19.03125" fill="black" rx="2"/><rect x="183" y="251.484375" width="85.84375" height="19.015625" fill="black" rx="2"/><rect x="70.203125" y="285.25" width="47.59375" height="21.515625" fill="black" rx="2"/><rect x="148.796875" y="285.25" width="50.40625" height="21.515625" fill="black" rx="2"/><rect x="228.953125" y="285.25" width="50.09375" height="21.515625" fill="black" rx="2"/><rect x="513" y="251.484375" width="85.84375" height="19.015625" fill="black" rx="2"/><rect x="400.203125" y="285.25" width="47.59375" height="21.515625" fill="black" rx="2"/><rect x="478.796875" y="285.25" width="50.40625" height="21.515625" fill="black" rx="2"/><rect x="558.953125" y="285.25" width="50.09375" height="21.515625" fill="black" rx="2"/><rect x="149.390625" y="344.078125" width="381.21875" height="19.015625" fill="black" rx="2"/></mask></defs>

  <!-- Action -->
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="240" y="20" width="200" height="56" rx="8" stroke-width="0.5" style="fill:rgb(250, 238, 218);stroke:rgb(133, 79, 11);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="340" y="42" text-anchor="middle" dominant-baseline="central" style="fill:rgb(99, 56, 6);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Action</text>
    <text x="340" y="60" text-anchor="middle" dominant-baseline="central" style="fill:rgb(133, 79, 11);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">e.g. count(), show(), write()</text>
  </g>

  <line x1="340" y1="76" x2="340" y2="104" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="352" y="93" dominant-baseline="central" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:start;dominant-baseline:central">triggers</text>

  <!-- Job -->
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="240" y="104" width="200" height="56" rx="8" stroke-width="0.5" style="fill:rgb(238, 237, 254);stroke:rgb(83, 74, 183);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="340" y="126" text-anchor="middle" dominant-baseline="central" style="fill:rgb(60, 52, 137);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Job</text>
    <text x="340" y="144" text-anchor="middle" dominant-baseline="central" style="fill:rgb(83, 74, 183);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">Full computation from action</text>
  </g>

  <line x1="340" y1="160" x2="340" y2="188" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="352" y="177" dominant-baseline="central" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:start;dominant-baseline:central">split by shuffle</text>

  <!-- Stage 1 -->
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="60" y="188" width="230" height="56" rx="8" stroke-width="0.5" style="fill:rgb(225, 245, 238);stroke:rgb(15, 110, 86);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="175" y="210" text-anchor="middle" dominant-baseline="central" style="fill:rgb(8, 80, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Stage 1</text>
    <text x="175" y="228" text-anchor="middle" dominant-baseline="central" style="fill:rgb(15, 110, 86);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">filter, map → shuffle write</text>
  </g>

  <!-- Stage 2 -->
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="390" y="188" width="230" height="56" rx="8" stroke-width="0.5" style="fill:rgb(225, 245, 238);stroke:rgb(15, 110, 86);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="505" y="210" text-anchor="middle" dominant-baseline="central" style="fill:rgb(8, 80, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Stage 2</text>
    <text x="505" y="228" text-anchor="middle" dominant-baseline="central" style="fill:rgb(15, 110, 86);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:central">shuffle read → aggregate</text>
  </g>

  <line x1="290" y1="160" x2="175" y2="186" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <line x1="390" y1="160" x2="505" y2="186" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" mask="url(#imagine-text-gaps-81iyo9)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>

  <!-- Tasks under Stage 1 -->
  <line x1="175" y1="244" x2="175" y2="272" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="187" y="261" dominant-baseline="central" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:start;dominant-baseline:central">1 per partition</text>

  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="60" y="272" width="68" height="40" rx="6" stroke-width="0.5" style="fill:rgb(241, 239, 232);stroke:rgb(95, 94, 90);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="94" y="296" text-anchor="middle" dominant-baseline="central" style="fill:rgb(68, 68, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Task 1</text>
  </g>
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="140" y="272" width="68" height="40" rx="6" stroke-width="0.5" style="fill:rgb(241, 239, 232);stroke:rgb(95, 94, 90);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="174" y="296" text-anchor="middle" dominant-baseline="central" style="fill:rgb(68, 68, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Task 2</text>
  </g>
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="220" y="272" width="68" height="40" rx="6" stroke-width="0.5" style="fill:rgb(241, 239, 232);stroke:rgb(95, 94, 90);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="254" y="296" text-anchor="middle" dominant-baseline="central" style="fill:rgb(68, 68, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Task 3</text>
  </g>

  <!-- Tasks under Stage 2 -->
  <line x1="505" y1="244" x2="505" y2="272" stroke="#3d3d3a" stroke-width="1" marker-end="url(#arrow)" style="fill:rgb(0, 0, 0);stroke:rgb(61, 61, 58);color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="517" y="261" dominant-baseline="central" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:start;dominant-baseline:central">1 per partition</text>

  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="390" y="272" width="68" height="40" rx="6" stroke-width="0.5" style="fill:rgb(241, 239, 232);stroke:rgb(95, 94, 90);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="424" y="296" text-anchor="middle" dominant-baseline="central" style="fill:rgb(68, 68, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Task 1</text>
  </g>
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="470" y="272" width="68" height="40" rx="6" stroke-width="0.5" style="fill:rgb(241, 239, 232);stroke:rgb(95, 94, 90);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="504" y="296" text-anchor="middle" dominant-baseline="central" style="fill:rgb(68, 68, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Task 2</text>
  </g>
  <g style="fill:rgb(0, 0, 0);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto">
    <rect x="550" y="272" width="68" height="40" rx="6" stroke-width="0.5" style="fill:rgb(241, 239, 232);stroke:rgb(95, 94, 90);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
    <text x="584" y="296" text-anchor="middle" dominant-baseline="central" style="fill:rgb(68, 68, 65);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:1;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:14px;font-weight:500;text-anchor:middle;dominant-baseline:central">Task 3</text>
  </g>

  <!-- Shuffle boundary label -->
  <line x1="60" y1="340" x2="620" y2="340" stroke="#73726c" stroke-width="0.5" stroke-dasharray="4 3" opacity="0.4" style="fill:rgb(0, 0, 0);stroke:rgb(115, 114, 108);color:rgb(0, 0, 0);stroke-width:0.5px;stroke-dasharray:4px, 3px;stroke-linecap:butt;stroke-linejoin:miter;opacity:0.4;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:16px;font-weight:400;text-anchor:start;dominant-baseline:auto"/>
  <text x="340" y="358" text-anchor="middle" opacity="0.6" style="fill:rgb(61, 61, 58);stroke:none;color:rgb(0, 0, 0);stroke-width:1px;stroke-linecap:butt;stroke-linejoin:miter;opacity:0.6;font-family:&quot;Anthropic Sans&quot;, -apple-system, BlinkMacSystemFont, &quot;Segoe UI&quot;, sans-serif;font-size:12px;font-weight:400;text-anchor:middle;dominant-baseline:auto">↑ shuffle boundary — stage 1 must complete before stage 2 starts</text>
</svg></div>
<script>try{window.frameElement.style.height=(document.body.scrollHeight+20)+"px";}catch(e){}</script>""")

In [ ]:
data = [(i, f"user_{i % 5}", i * 10) for i in range(1, 21)]
df = spark.createDataFrame(data, ["id", "user", "amount"])

# getNumPartitions via RDD is not available on serverless — use try/except
try:
    print(f"Partitions: {df.rdd.getNumPartitions()}")
except Exception:
    print("Note: .rdd.getNumPartitions() not available on serverless.")
    print("On a classic cluster this returns the number of data partitions.")

# This groupBy triggers a shuffle → 2 stages — works on all compute types
result = df.groupBy("user").sum("amount")
result.show()
# Open the Spark UI (cluster → Spark UI tab) to see the 2 stages

## 3. Narrow vs Wide Transformations

This is a core exam concept — know the difference cold.

| | Narrow | Wide |
|---|---|---|
| **Data movement** | None — each partition is processed independently | Shuffle — data moves across the network |
| **Stage boundary** | No | Yes — creates a new stage |
| **Examples** | `filter`, `select`, `withColumn`, `map`, `union` | `groupBy`, `join`, `distinct`, `repartition`, `orderBy` |
| **Performance** | Fast | Slower (network I/O) |

```
NARROW (filter)              WIDE (groupBy)

P0 ──filter──► P0'          P0 ─┐
P1 ──filter──► P1'          P1 ─┼──shuffle──► P0' (grouped)
P2 ──filter──► P2'          P2 ─┘             P1' (grouped)

No data crosses partitions   Data redistributed across network
```

In [ ]:
from pyspark.sql.functions import col

# Narrow — filter stays within each partition, no shuffle
narrow = df.filter(col("amount") > 50).select("id", "amount")
narrow.explain()  # 1 stage expected

# Wide — groupBy requires data from all partitions to be redistributed
wide = df.groupBy("user").count()
wide.explain()  # 2 stages: scan+shuffle write, then shuffle read+aggregate

## 4. Deployment Modes

| Mode | Driver location | Typical use |
|---|---|---|
| **local** | Same machine as executor | Local dev/testing — `local[*]` uses all cores |
| **client** | Machine submitting the job | Interactive notebooks (Databricks, Jupyter) |
| **cluster** | A worker node in the cluster | Production batch jobs (`spark-submit`) |

> **Exam tip:** In **client mode**, if the submitting machine fails, the Driver dies and the job fails.  
> In **cluster mode**, the Driver runs inside the cluster — the submitting machine can disconnect safely.

In Databricks, you always run in **client mode** from the notebook.

In [ ]:
# Verify current mode and config
print(spark.conf.get("spark.submit.deployMode", "client (default)"))
print(spark.conf.get("spark.executor.memory", "not set"))
print(spark.conf.get("spark.executor.cores", "not set"))

## 5. Fault Tolerance via Lineage

Spark does **not** replicate data between steps. Instead it remembers the **lineage** — the full sequence of transformations that produced each partition.

If an executor dies mid-job:
1. The Cluster Manager detects the failure
2. Spark re-schedules only the **failed tasks** on a healthy executor
3. The lineage is replayed from the last checkpoint or from the source

> **Exam tip:** Spark's fault tolerance model is **lineage-based recomputation**, not replication. RDDs store the recipe, not copies of the data.

In [ ]:
# toDebugString shows RDD lineage — not available on serverless
try:
    rdd = df.filter(col("amount") > 50).select("user").rdd
    print(rdd.toDebugString().decode())
except Exception:
    print("Note: .rdd and toDebugString() not available on serverless compute.")
    print("On a classic cluster this prints the full lineage chain, e.g.:")
    print("  MapPartitionsRDD -> FilteredRDD -> MapPartitionsRDD -> ParallelCollectionRDD")
    print()
    print("The concept still applies on serverless — use explain() to see the logical lineage:")
    df.filter(col("amount") > 50).select("user").explain()

## 6. End-to-End Walkthrough: Spark on Kubernetes

Trace a real query through a 4-node Kubernetes cluster processing 120 GB of bank transactions — from partition creation to final write. Step through each phase using the buttons.

In [ ]:
displayHTML("""<style>
html,body{min-height:900px;}
</style>

<style>
  .sr-only{position:absolute;width:1px;height:1px;overflow:hidden;clip:rect(0,0,0,0)}
  .step-dot{width:10px;height:10px;border-radius:50%;background:#e0ded6;transition:background .2s}
  .step-dot.active{background:#1a56db}
  .nav-btn{padding:7px 16px;border:0.5px solid #e0ded6;border-radius:8px;background:#ffffff;color:#1a1a1a;cursor:pointer;font-size:13px;font-family:inherit}
  .nav-btn:hover{background:#f5f4ed}
  .nav-btn:disabled{opacity:0.35;cursor:default}
  .badge{display:inline-block;font-size:11px;font-weight:500;padding:2px 8px;border-radius:99px;margin-right:4px}
  .b-purple{background:#EEEDFE;color:#3C3489}
  .b-teal{background:#E1F5EE;color:#085041}
  .b-coral{background:#FAECE7;color:#712B13}
  .b-amber{background:#FAEEDA;color:#633806}
  .b-gray{background:#F1EFE8;color:#444441}
  .node-box{border:0.5px solid #e0ded6;border-radius:8px;padding:8px 10px;font-size:12px;background:#ffffff}
  .highlight{border-color:#93b4f8!important;background:#f0f4ff!important}
  .dim{opacity:0.35}
  .row{display:flex;gap:8px;align-items:flex-start;flex-wrap:wrap}
  .col{display:flex;flex-direction:column;gap:6px}
  .label{font-size:11px;color:#999990;margin-bottom:2px}
  .mono{font-family:ui-monospace,'Cascadia Code','Courier New',monospace;font-size:11px;color:#666660}
</style>

<h2 class="sr-only">Step-by-step walkthrough of Spark on a 4-node Kubernetes cluster processing a bank transactions table</h2>

<div style="padding:1rem 0">

  <div id="step-dots" style="display:flex;gap:8px;margin-bottom:1.25rem;align-items:center">
    <span style="font-size:12px;color:#999990;margin-right:4px">step</span>
  </div>

  <div id="stage-label" style="font-size:13px;color:#666660;margin-bottom:6px"></div>
  <div id="stage-title" style="font-size:17px;font-weight:500;color:#1a1a1a;margin-bottom:10px"></div>
  <div id="stage-desc" style="font-size:14px;color:#666660;line-height:1.65;margin-bottom:1.25rem;max-width:600px"></div>

  <div id="diagram" style="margin-bottom:1.25rem"></div>

  <div id="detail-box" style="border-left:2px solid #93b4f8;padding:8px 14px;font-size:13px;color:#666660;border-radius:0 8px 8px 0;margin-bottom:1.25rem;display:none"></div>

  <div style="display:flex;align-items:center;gap:10px;margin-top:8px">
    <button class="nav-btn" id="btn-prev" onclick="go(-1)">← prev</button>
    <button class="nav-btn" id="btn-next" onclick="go(1)">next →</button>
    
  </div>
</div>

<script>
const steps = [
  {
    label: "setup",
    title: "The cluster: 1 Driver + 3 workers",
    desc: "Kubernetes schedules 4 pods. One pod runs the Spark Driver (your SparkSession). The other three are Worker pods — each gets a slice of CPU and RAM as Executors. Your bank transaction table is a Parquet file on S3: 120 GB, ~800 million rows.",
    detail: "In Kubernetes mode the Driver itself runs as a pod. The Cluster Manager is Kubernetes — it launches Executor pods on request and restarts them if they crash.",
    diag: () => {
      return `
      <div class="row" style="align-items:stretch">
        <div class="col" style="min-width:130px">
          <div class="label">Driver pod</div>
          <div class="node-box highlight" style="flex:1">
            <div class="badge b-purple">Driver</div>
            <div style="margin-top:6px;font-size:12px;color:#666660">SparkSession<br>DAG builder<br>Task scheduler</div>
          </div>
        </div>
        <div style="display:flex;flex-direction:column;justify-content:center;padding:0 6px;color:#999990;font-size:18px">→</div>
        <div class="col" style="flex:1">
          <div class="label">Worker pods (Executors)</div>
          <div class="row" style="flex:1">
            ${[1,2,3].map(i=>`<div class="node-box" style="flex:1;min-width:100px">
              <div class="badge b-coral">Worker ${i}</div>
              <div style="margin-top:6px" class="mono">8 cores<br>32 GB RAM</div>
            </div>`).join('')}
          </div>
        </div>
      </div>
      <div style="margin-top:10px;font-size:12px;color:#999990">
        <span class="badge b-gray">S3</span> transactions.parquet — 120 GB, ~800M rows
      </div>`
    }
  },
  {
    label: "reading",
    title: "Spark reads the file — partitions are born",
    desc: "Spark asks S3 how many 128 MB Parquet row groups the file has. It gets back ~960 row groups. Each becomes one partition. Spark creates 960 tasks in Stage 1 — one per partition. No data has moved yet. This is still lazy.",
    detail: "Spark's default partition size is 128 MB (spark.sql.files.maxPartitionBytes). 120 GB ÷ 128 MB ≈ 960 partitions. Each partition holds roughly 800K rows.",
    diag: () => `
      <div style="margin-bottom:8px;font-size:12px;color:#999990">960 partitions created from S3 row groups</div>
      <div style="display:flex;flex-wrap:wrap;gap:4px;max-width:580px">
        ${Array.from({length:40},(_,i)=>`<div style="width:22px;height:22px;border-radius:4px;background:${i<5?'#E1F5EE':i<10?'#E1F5EE':'#F1EFE8'};border:0.5px solid ${i<10?'#0F6E56':'#D3D1C7'};font-size:9px;display:flex;align-items:center;justify-content:center;color:${i<10?'#085041':'#888780'}">${i+1}</div>`).join('')}
        <div style="width:22px;height:22px;border-radius:4px;background:#F1EFE8;border:0.5px solid #D3D1C7;font-size:9px;display:flex;align-items:center;justify-content:center;color:#888780">…</div>
      </div>
      <div style="margin-top:8px;font-size:12px;color:#999990">showing 40 of 960 &nbsp;·&nbsp; each = ~128 MB = ~800K rows</div>`
  },
  {
    label: "stage1",
    title: "Stage 1: tasks run in parallel across workers",
    desc: "The Driver sends tasks to the 3 Executors. Each Executor has 8 cores → 8 concurrent tasks. All 3 workers together run 24 tasks at a time. 960 tasks ÷ 24 concurrent = ~40 rounds. Each task reads its partition from S3, applies your filter (e.g. amount > 10000), and holds the result in memory.",
    detail: "Task = partition + transformation logic. The Driver serializes the closure (your filter function) and ships it to each Executor. Data never goes through the Driver — Executors pull directly from S3.",
    diag: () => `
      <div class="row" style="align-items:flex-start">
        ${[
          {w:'Worker 1', tasks:['T1','T2','T3','T4','T5','T6','T7','T8'], color:'b-coral'},
          {w:'Worker 2', tasks:['T9','T10','T11','T12','T13','T14','T15','T16'], color:'b-coral'},
          {w:'Worker 3', tasks:['T17','T18','T19','T20','T21','T22','T23','T24'], color:'b-coral'},
        ].map(w=>`
          <div class="col" style="flex:1;min-width:120px">
            <div class="label">${w.w}</div>
            <div style="display:flex;flex-wrap:wrap;gap:4px">
              ${w.tasks.map(t=>`<div class="node-box" style="padding:4px 7px;font-size:11px"><span class="badge ${w.color}" style="margin:0">${t}</span></div>`).join('')}
            </div>
            <div class="mono" style="margin-top:4px">8 concurrent tasks</div>
          </div>`).join('')}
      </div>
      <div style="margin-top:10px;font-size:12px;color:#999990">round 1 of ~40 · each task: read partition from S3 → filter rows → keep in memory</div>`
  },
  {
    label: "shuffle",
    title: "Stage boundary: the shuffle",
    desc: "You call groupBy('customer_id').sum('amount'). Rows for the same customer_id are scattered across all 960 partitions on all 3 workers. Before Stage 2 can aggregate, every row must travel to the same worker as its customer_id. This is the shuffle.",
    detail: "Spark hashes each customer_id modulo the number of output partitions (default 200). All rows with the same hash go to the same Executor. This produces network traffic proportional to your data size — on 120 GB that is significant.",
    diag: () => `
      <div class="row" style="align-items:flex-start;gap:12px">
        <div class="col" style="flex:1">
          <div class="label">Before shuffle (stage 1 output)</div>
          ${[1,2,3].map(w=>`
            <div class="node-box" style="margin-bottom:4px">
              <span class="badge b-coral">Worker ${w}</span>
              <span class="mono"> cust_A, cust_C, cust_B, cust_A, cust_D …</span>
            </div>`).join('')}
        </div>
        <div style="display:flex;flex-direction:column;justify-content:center;align-items:center;padding:0 8px;gap:4px">
          <div style="font-size:18px;color:#999990">⇄</div>
          <div style="font-size:11px;color:#991b1b;font-weight:500">shuffle</div>
          <div style="font-size:11px;color:#999990">network</div>
        </div>
        <div class="col" style="flex:1">
          <div class="label">After shuffle (stage 2 input)</div>
          <div class="node-box" style="margin-bottom:4px"><span class="badge b-teal">Worker 1</span><span class="mono"> all cust_A, all cust_D …</span></div>
          <div class="node-box" style="margin-bottom:4px"><span class="badge b-teal">Worker 2</span><span class="mono"> all cust_B, all cust_E …</span></div>
          <div class="node-box"><span class="badge b-teal">Worker 3</span><span class="mono"> all cust_C, all cust_F …</span></div>
        </div>
      </div>
      <div style="margin-top:10px;font-size:12px;color:#991b1b">stage 1 must fully complete before stage 2 starts — this is the stage boundary</div>`
  },
  {
    label: "stage2",
    title: "Stage 2: aggregate — one task per output partition",
    desc: "Spark creates 200 output partitions by default (spark.sql.shuffle.partitions). Stage 2 has 200 tasks. Each task reads its slice of the shuffled data from disk/memory and sums the amounts per customer_id. Results are small — one row per customer.",
    detail: "200 tasks across 24 concurrent slots = ~9 rounds. The output is much smaller than the input — aggregation is a massive reduction. This result can then be collected to the Driver or written back to S3.",
    diag: () => `
      <div style="margin-bottom:8px;font-size:12px;color:#999990">200 output partitions, 24 concurrent tasks</div>
      <div style="display:flex;flex-wrap:wrap;gap:4px;max-width:500px">
        ${Array.from({length:24},(_,i)=>`<div class="node-box highlight" style="padding:4px 8px;font-size:11px">T${i+1}</div>`).join('')}
        <div style="font-size:12px;align-self:center;color:#999990;padding:4px">… +176 more</div>
      </div>
      <div style="margin-top:12px">
        <div class="label">each task produces</div>
        <div class="node-box" style="display:inline-block;margin-top:4px">
          <span class="mono">customer_id | total_amount</span><br>
          <span class="mono">cust_A &nbsp;&nbsp;&nbsp; | 284,500.00</span><br>
          <span class="mono">cust_D &nbsp;&nbsp;&nbsp; | 91,200.00</span><br>
          <span class="mono">… (slice of customers)</span>
        </div>
      </div>`
  },
  {
    label: "result",
    title: "Job complete — result back to Driver or S3",
    desc: "The 200 output partitions are either written to S3 as Parquet (df.write) — staying on the workers — or collected to the Driver (df.collect()) where they're assembled into a Python list. For 800M rows of transactions, you almost never collect — you write.",
    detail: "collect() sends all data over the network to the Driver JVM and then to your Python process. On 800M rows that kills the Driver. write() keeps data distributed and streams it straight from Executors to S3 in parallel — much faster and safer.",
    diag: () => `
      <div class="row" style="gap:16px;align-items:flex-start">
        <div class="col" style="flex:1">
          <div class="label" style="color:#065f46">✓ write() — recommended</div>
          <div class="node-box" style="border-color:#6ee7b7">
            <div class="mono">df.write.parquet("s3://results/")</div>
            <div style="margin-top:6px;font-size:12px;color:#666660">Workers stream directly to S3<br>Driver not involved<br>Parallelism preserved</div>
          </div>
        </div>
        <div class="col" style="flex:1">
          <div class="label" style="color:#991b1b">✗ collect() — dangerous at scale</div>
          <div class="node-box" style="border-color:#fca5a5">
            <div class="mono">df.collect()</div>
            <div style="margin-top:6px;font-size:12px;color:#666660">All data → Driver JVM → Python<br>Driver OOM on large tables<br>Kills the job</div>
          </div>
        </div>
      </div>
      <div style="margin-top:12px;padding:8px 12px;border-radius:8px;background:#f5f4ed;font-size:12px;color:#666660">
        Full job: 1 action → 1 job → 2 stages → 1160 tasks total (960 + 200)
      </div>`
  }
];

let cur = 0;

function renderDots() {
  const el = document.getElementById('step-dots');
  const children = Array.from(el.children).filter(c => c.classList.contains('step-dot'));
  if (children.length === 0) {
    steps.forEach((_,i) => {
      const d = document.createElement('div');
      d.className = 'step-dot' + (i===cur?' active':'');
      el.appendChild(d);
    });
    const total = document.createElement('span');
    total.id = 'dot-label';
    total.style.cssText = 'font-size:12px;color:#999990;margin-left:6px';
    el.appendChild(total);
  } else {
    children.forEach((d,i) => d.classList.toggle('active', i===cur));
  }
  const lbl = document.getElementById('dot-label');
  if (lbl) lbl.textContent = (cur+1) + ' / ' + steps.length;
}

function render() {
  const s = steps[cur];
  document.getElementById('stage-label').textContent = s.label.toUpperCase();
  document.getElementById('stage-title').textContent = s.title;
  document.getElementById('stage-desc').textContent = s.desc;
  document.getElementById('diagram').innerHTML = s.diag();
  const db = document.getElementById('detail-box');
  if (s.detail) { db.style.display='block'; db.textContent = s.detail; }
  else db.style.display='none';
  document.getElementById('btn-prev').disabled = cur===0;
  document.getElementById('btn-next').disabled = cur===steps.length-1;
  renderDots();
}

function go(dir) { cur = Math.max(0, Math.min(steps.length-1, cur+dir)); render(); }
render();
try{window.frameElement.style.height=(document.body.scrollHeight+20)+"px";}catch(e){}
</script>
""")

---

## Day 2 Checklist

- [ ] Can draw the Driver / Cluster Manager / Executor triangle from memory
- [ ] Understand: Action → Job → Stages → Tasks
- [ ] Know which transformations are narrow and which are wide
- [ ] Can explain why groupBy creates a stage boundary (shuffle)
- [ ] Know the 3 deployment modes and when each is used
- [ ] Understand fault tolerance via lineage recomputation

**Next:** Day 3 — Lazy Evaluation & the Catalyst Optimizer

---

## Interactive Quiz — Day 2 Architecture

Five topic areas, three questions each. Pick a topic, answer the questions, then move on. Run the cell below to launch the quiz.

In [ ]:
displayHTML("""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Spark Fundamentals Quiz</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  html, body { min-height: 700px; }
  body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #fff; color: #1a1a1a; padding: 1.5rem; max-width: 720px; }
  h1 { font-size: 18px; font-weight: 500; margin-bottom: 1.25rem; color: #1a1a1a; }
  .topic-bar { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 1.5rem; }
  .topic-btn { padding: 7px 13px; border: 1px solid #ccc; border-radius: 8px; background: #fff; color: #1a1a1a; cursor: pointer; font-size: 13px; font-family: inherit; }
  .topic-btn:hover { background: #f5f5f5; }
  .topic-btn.active { background: #e8f0fe; color: #1a56db; border-color: #93b4f8; }
  .meta { display: flex; justify-content: space-between; margin-bottom: 1rem; }
  .meta span { font-size: 13px; color: #666; }
  .question { font-size: 15px; font-weight: 500; margin-bottom: 1rem; line-height: 1.5; }
  .opt-btn { width: 100%; text-align: left; padding: 10px 14px; margin: 6px 0; border: 1px solid #ddd; border-radius: 8px; background: #fff; color: #1a1a1a; cursor: pointer; font-size: 14px; font-family: inherit; line-height: 1.4; }
  .opt-btn:hover { background: #f5f5f5; }
  .opt-btn.correct { background: #ecfdf5 !important; border-color: #6ee7b7 !important; color: #065f46 !important; pointer-events: none; }
  .opt-btn.wrong { background: #fef2f2 !important; border-color: #fca5a5 !important; color: #991b1b !important; pointer-events: none; }
  .opt-btn.locked { pointer-events: none; }
  .explanation { padding: 10px 14px; border-left: 3px solid #93b4f8; font-size: 14px; color: #444; margin-top: 1rem; line-height: 1.6; background: #f8faff; border-radius: 0 6px 6px 0; }
  .nav { margin-top: 1rem; }
  .score-line { font-size: 14px; color: #555; margin-bottom: 8px; }
  .placeholder { font-size: 15px; color: #888; }
</style>
</head>
<body>

<h1>Spark fundamentals quiz</h1>
<div class="topic-bar" id="topic-bar"></div>
<div id="quiz-area"><p class="placeholder">Pick a topic above to start.</p></div>

<script>
const topics = [
  {
    id: "arch", label: "Driver / Cluster / Executor",
    questions: [
      { q: "Which component breaks a Spark application into tasks and schedules them?", opts: ["Cluster Manager","Driver","Executor","Worker Node"], ans: 1, exp: "The Driver runs your main() function, creates the SparkContext, builds the DAG, and schedules tasks onto Executors." },
      { q: "What does the Cluster Manager do?", opts: ["Runs transformations on data","Allocates resources (CPU/memory) across the cluster","Stores the lineage graph","Serializes DataFrames"], ans: 1, exp: "The Cluster Manager (YARN, Mesos, Kubernetes, Standalone) negotiates resources. It does not touch your data." },
      { q: "Where does actual data processing happen?", opts: ["Driver JVM","Cluster Manager","Executors","SparkContext"], ans: 2, exp: "Executors are JVM processes on Worker nodes. They run tasks, cache data, and report results back to the Driver." }
    ]
  },
  {
    id: "jobstages", label: "Action to Job to Stages to Tasks",
    questions: [
      { q: "What triggers Spark to actually execute a computation?", opts: ["A transformation like map()","An action like count() or show()","Creating a DataFrame","Calling explain()"], ans: 1, exp: "Transformations are lazy -- they just build a plan. An action like collect(), count(), show(), or write() triggers execution." },
      { q: "What creates a stage boundary in Spark?", opts: ["Any transformation","A narrow transformation","A shuffle (wide transformation)","Calling cache()"], ans: 2, exp: "Shuffles require data to move between partitions across the network -- Spark must complete Stage N before starting Stage N+1." },
      { q: "How does a Task relate to a Stage?", opts: ["One Task per Job","One Task per Stage","One Task per partition within a Stage","One Task per Executor"], ans: 2, exp: "A Stage is split into Tasks -- one per partition. 200 partitions means 200 Tasks, each running on one Executor slot." }
    ]
  },
  {
    id: "transformations", label: "Narrow vs Wide",
    questions: [
      { q: "Which of these is a narrow transformation?", opts: ["groupBy()","join() on two large tables","filter()","distinct()"], ans: 2, exp: "filter() keeps each row in its current partition -- no data moves. groupBy, join, and distinct all require shuffles." },
      { q: "Why is groupBy() a wide transformation?", opts: ["It creates a new DataFrame","Rows with the same key may live in different partitions and must be co-located","It always reads from disk","It uses more memory"], ans: 1, exp: "To aggregate by a key, all rows for that key must land on the same executor. Since they are scattered, a shuffle is needed." },
      { q: "Which set contains only narrow transformations?", opts: ["map, filter, select","groupBy, join, repartition","distinct, orderBy, join","map, groupBy, filter"], ans: 0, exp: "map, filter, and select operate partition-by-partition with no cross-partition data movement. All others trigger shuffles." }
    ]
  },
  {
    id: "deploy", label: "Deployment modes",
    questions: [
      { q: "In client mode, where does the Driver run?", opts: ["On a worker node inside the cluster","On the machine that submitted the job","On the Cluster Manager","On the first Executor"], ans: 1, exp: "Client mode: Driver stays on your local machine. Good for interactive notebooks. If your machine dies, the job dies." },
      { q: "In cluster mode, what happens to the Driver?", opts: ["It runs on the submitting machine","It is deployed inside the cluster on a worker node","It merges with the Cluster Manager","It is not used"], ans: 1, exp: "Cluster mode: Driver runs inside the cluster. The submitting process exits after launch. Best for production batch jobs." },
      { q: "When would you use local mode?", opts: ["Production ETL pipelines","Multi-node distributed jobs","Development, testing, and learning on a single machine","When the cluster is down for maintenance"], ans: 2, exp: "local[*] runs Driver and Executors in the same JVM process. No cluster needed -- great for development and unit tests." }
    ]
  },
  {
    id: "fault", label: "Fault tolerance",
    questions: [
      { q: "How does Spark recover from a lost partition without rerunning everything?", opts: ["Restores from a checkpoint on HDFS","Uses lineage to recompute only the lost partition","Asks another Executor to copy its partition","Re-reads all source data from scratch"], ans: 1, exp: "Spark tracks the DAG of transformations (lineage). If a partition is lost, it replays only the transformations needed to rebuild it." },
      { q: "What is the lineage graph?", opts: ["A log of completed tasks","A record of which Executors ran which tasks","A DAG showing how each partition was derived from transformations","The physical execution plan after optimization"], ans: 2, exp: "The lineage (DAG) records every transformation applied to produce each partition -- the recipe Spark uses to rebuild lost data." },
      { q: "When would you explicitly checkpoint a DataFrame?", opts: ["Before every action","When the lineage chain becomes very long, to truncate recomputation cost","Only with streaming jobs","When switching from client to cluster mode"], ans: 1, exp: "After many chained transformations, recomputing from scratch gets expensive. Checkpointing saves a snapshot and truncates the lineage." }
    ]
  }
];

const scores = {};
topics.forEach(function(t) { scores[t.id] = { correct: 0, total: 0 }; });
let activeTopic = null;
let activeQ = 0;
let answered = false;

function renderTopicBar() {
  const bar = document.getElementById("topic-bar");
  bar.innerHTML = "";
  topics.forEach(function(t) {
    const s = scores[t.id];
    const btn = document.createElement("button");
    btn.className = "topic-btn" + (activeTopic === t.id ? " active" : "");
    btn.textContent = t.label + (s.total > 0 ? " " + s.correct + "/" + s.total : "");
    btn.addEventListener("click", function() {
      activeTopic = t.id; activeQ = 0; answered = false; renderAll();
    });
    bar.appendChild(btn);
  });
}

function renderQuestion() {
  const area = document.getElementById("quiz-area");
  if (!activeTopic) {
    area.innerHTML = "<p class='placeholder'>Pick a topic above to start.</p>";
    return;
  }
  const topic = topics.find(function(t) { return t.id === activeTopic; });
  const q = topic.questions[activeQ];
  area.innerHTML =
    "<div class='meta'><span>" + topic.label + "</span><span>" + (activeQ + 1) + " / " + topic.questions.length + "</span></div>" +
    "<p class='question'>" + q.q + "</p>" +
    "<div id='opts-wrap'></div>" +
    "<div id='exp-wrap'></div>" +
    "<div id='nav-wrap' class='nav'></div>";

  const wrap = document.getElementById("opts-wrap");
  q.opts.forEach(function(opt, i) {
    const btn = document.createElement("button");
    btn.className = "opt-btn";
    btn.textContent = opt;
    btn.addEventListener("click", function() { handleAnswer(i); });
    wrap.appendChild(btn);
  });
}

function handleAnswer(chosen) {
  if (answered) return;
  answered = true;
  const topic = topics.find(function(t) { return t.id === activeTopic; });
  const q = topic.questions[activeQ];
  scores[activeTopic].total++;
  if (chosen === q.ans) scores[activeTopic].correct++;

  document.querySelectorAll(".opt-btn").forEach(function(b, i) {
    b.classList.add("locked");
    if (i === q.ans) b.classList.add("correct");
    else if (i === chosen) b.classList.add("wrong");
  });

  document.getElementById("exp-wrap").innerHTML =
    "<div class='explanation'>" + q.exp + "</div>";

  // resize iframe to fit new content
  try { window.frameElement.style.height = document.body.scrollHeight + 20 + "px"; } catch(e) {}

  const nav = document.getElementById("nav-wrap");
  if (activeQ < topic.questions.length - 1) {
    const nb = document.createElement("button");
    nb.className = "topic-btn";
    nb.textContent = "Next question";
    nb.addEventListener("click", function() { activeQ++; answered = false; renderAll(); });
    nav.appendChild(nb);
  } else {
    const s = scores[activeTopic];
    nav.innerHTML = "<p class='score-line'>Topic done -- score: <strong>" + s.correct + "/" + s.total + "</strong></p>";
    const rb = document.createElement("button");
    rb.className = "topic-btn";
    rb.textContent = "Retry topic";
    rb.addEventListener("click", function() { scores[activeTopic] = { correct: 0, total: 0 }; activeQ = 0; answered = false; renderAll(); });
    nav.appendChild(rb);
  }
  renderTopicBar();
}

function renderAll() {
  renderTopicBar();
  renderQuestion();
  try { window.frameElement.style.height = document.body.scrollHeight + 20 + "px"; } catch(e) {}
}

renderAll();
</script>
</body>
</html>""")